# Task 1.3: Convolution and Deconvolution

**Student Name:** Haris Salii  
**Country:** Switzerland  
**Semester term:** HS26


## Day 11 Data & Domain

### Use Case

Für die kurzfristige Betriebsplanung einer Photovoltaikanlage im Raum Bern soll ein geglätteter Verlauf der Globalstrahlung den allgemeinen Einstrahlungstrend sichtbar machen. Betreiber können diesen Verlauf für Batterie- und Einspeiseentscheidungen nutzen. Der Anwendungsfall ist für die Schweiz relevant, weil der wachsende Solarstromanteil wetterabhängige und kurzfristig schwankende Einspeisung erzeugt.

### Problem Statement

Die 10-Minuten-Werte enthalten starke aufeinanderfolgende Änderungen, welche den allgemeinen Verlauf überdecken können. Eine Faltung mit einem zu breiten Glättungskern kann jedoch relevante Einbrüche und Spitzen abschwächen. Die Entfaltung soll deshalb untersuchen, wie weit sich die ursprüngliche Struktur aus dem geglätteten Signal rekonstruieren lässt und wann dabei Störungen verstärkt werden.

### Experimental Objective

Untersucht wird der Zielkonflikt zwischen einem ruhigeren Verlauf und dem Erhalt kurzfristiger Strahlungsänderungen. Für die PV-Planung ist entscheidend, ob eine stabile Rekonstruktion die durch die Glättung abgeschwächte Struktur wieder annähert.

### Data Definition and Source

Verwendet wird die Globalstrahlung `gre000z0` der MeteoSwiss-Station Bern/Zollikofen (`BER`) in **W/m²**. Die Werte sind gleichmässig erfasste **10-Minuten-Mittelwerte** aus dem Datensatz *Automatic Weather Stations Measurement Values*. Das einzige Untersuchungsfenster umfasst den **21. Juli 2025 von 09:00 bis 15:00 Uhr** und enthält 37 aufeinanderfolgende Werte ohne fehlende oder doppelte Zeitstempel. Der Abschnitt ist geeignet, weil er einen Tagestrend und mehrere kurze Einbrüche und Erholungen im selben Signal zeigt.

Quelle: MeteoSwiss, Automatic Weather Stations Measurement Values, Station `BER`, Parameter `gre000z0`: https://opendatadocs.meteoswiss.ch/de/a-data-groundbased/a1-automatic-weather-stations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("../data/solarstrahlung_bern_2025.csv", sep=";")
data["time"] = pd.to_datetime(data["reference_timestamp"], dayfirst=True)

window = data[
    (data["time"] >= "2025-07-21 09:00")
    & (data["time"] <= "2025-07-21 15:00")
].copy()

print("Anzahl Werte:", len(window))
print("Fehlende Strahlungswerte:", window["gre000z0"].isna().sum())
print("Doppelte Zeitstempel:", window["time"].duplicated().sum())

plt.figure(figsize=(11, 4))
plt.plot(window["time"], window["gre000z0"], marker="o", markersize=3)
plt.xlabel("Uhrzeit")
plt.ylabel("Globalstrahlung [W/m²]")
plt.title("Globalstrahlung in Bern/Zollikofen am 21. Juli 2025")
plt.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Observation

Zwischen 09:00 und 15:00 Uhr ist kein gleichmässiger Verlauf sichtbar. Auf starke Anstiege folgen mehrere deutliche Einbrüche, besonders im Bereich um 10:00–11:00 Uhr und nach 12:20 Uhr. Damit enthält das Fenster genau die kurzfristige Struktur, deren Abschwächung durch die Faltung und deren teilweise Wiederherstellung durch die Entfaltung untersucht werden soll. Die Messwerte sind bereits über zehn Minuten gemittelt; schnellere Änderungen innerhalb eines Intervalls sind nicht enthalten.

## Day 12 Methodological Design

### Theoretical Foundation and Method Choice

Die diskrete Faltung berechnet jeden Ausgabewert als gewichtete Summe benachbarter Signalwerte:

$$y[n] = \sum_k h[k]\,x[n-k].$$

Ein symmetrischer gleitender Mittelwert ist passend, weil er schnelle Änderungen dämpft, die Summe seiner Gewichte aber 1 beträgt und das mittlere Strahlungsniveau dadurch erhalten bleibt. Verglichen werden ein 3-Punkte-Kern $h_3=[1/3,1/3,1/3]$ und ein 5-Punkte-Kern $h_5=[1/5,\ldots,1/5]$. Der erste glättet über 30 Minuten, der zweite über 50 Minuten.

Für die Entfaltung wurden drei Ansätze betrachtet. Eine direkte Inversion kann bei einem störungsfreien, exakt bekannten Filter gute Resultate liefern, reagiert aber empfindlich auf kleine Störungen. Ein Wiener-Filter berücksichtigt ein Signal- und Rauschmodell, für dieses einzelne Tagesfenster liegt jedoch kein verlässliches Leistungsspektrum des Rauschens vor. Gewählt wird deshalb eine **Tikhonov-regularisierte Inversion**, weil ihre Stabilisierung über einen sichtbaren Parameter $\lambda$ gesteuert werden kann:

$$\hat{x}=(H^T H+\lambda I)^{-1}H^T y.$$

Dabei beschreibt $H$ die Faltung als Matrix. Grössere Werte von $\lambda$ begrenzen die Verstärkung von Störungen stärker, erzeugen aber mehr Abweichung vom Original.

### Parameters and Performance Requirement

Die Abtastzeit beträgt 10 Minuten. Als Baseline dienen der 3-Punkte-Kern, Randfortsetzung mit dem jeweils äussersten Messwert und $\lambda=0{,}01$. Der Randmodus erhält die Signallänge von 37 Werten, kann die ersten und letzten Werte aber beeinflussen. Für eine faire Stabilitätsprüfung wird bei allen Konfigurationen dieselbe reproduzierbare additive Störung mit $\sigma=20\,\mathrm{W/m^2}$ verwendet. Sie entspricht ungefähr zwei Prozent der sichtbaren Tagesskala und ist ein kontrollierter Robustheitstest, kein Modell des tatsächlichen MeteoSwiss-Messfehlers.

Die Rekonstruktion gilt für diesen Versuch als praktisch hilfreich, wenn ihr RMSE gegenüber dem Original kleiner ist als der RMSE des geglätteten Signals und keine deutlich unplausiblen Ausschläge entstehen. Diese Aussage bezieht sich nur auf die Wiederherstellung der hier bekannten 10-Minuten-Werte.

### Experimental Design

Die Kernelgrösse wird systematisch zwischen 3 und 5 Punkten variiert. Für die Regularisierung werden $\lambda\in\{0;0{,}001;0{,}01;0{,}1\}$ verglichen; $\lambda=0$ entspricht der direkten Inversion. Das Zeitfenster, die Randbehandlung und die Störung bleiben konstant. Erwartet wird, dass der 5-Punkte-Kern stärker glättet und dass eine mittlere Regularisierung bei gestörten Daten stabiler rekonstruiert als die direkte Inversion.

### Methodological Limitations

Das Verfahren nimmt einen linearen, exakt bekannten Filter an. In realen Messketten können unbekannte Störungen, andere Filter und nichtlineare Effekte diese Annahme verletzen. Ausserdem kann keine Entfaltung Änderungen wiederherstellen, die bereits durch die ursprüngliche 10-Minuten-Mittelung verloren gingen.


## Day 13 Implementation


## Day 14 Evaluation

## Day 15 Analysis & Communication